# SSD MobileNet V2 FPNLite (1024×1024) — PhenoBench finetune (session 2)

Fine-tunes **SSD MobileNet V2 FPNLite (1024×1024)** for **multi-class crop-vs-weed instance
detection** on the PhenoBench dataset, and publishes a self-contained
`finetune/` artifact (the fp32 base model used downstream for post-training
quantization and as the QAT starting point). The dedicated `ptq/` directory is
reserved for the companion PTQ (QAT-parity) runs.

- **Detector:** a single-shot detector (SSD) head with a lightweight feature-pyramid network (FPNLite) on a MobileNetV2 backbone.
- **Framework:** TensorFlow Object Detection API (TF 2.11), fine-tuned from the
  COCO-2017 checkpoint.
- **Task:** two foreground classes (`crop`, `weed`); detecting just the `weed` class is the single-class (`sc`) variant, not this one.

All training, curve and export plumbing is delegated to the shared
`agri_vision_edge.tfod_trainer` package, so this notebook only carries the
experiment-specific pieces: the `FineTuneConfig`, the `ExperimentManifest`,
publication-quality curves, and qualitative evaluation. The same run can be
driven head-less from Python — see `notebooks/finetuning.py` for an interactive
marimo front-end over the identical `FinetuneRunConfig` + `run_finetune`.

## Resolution ladder

This notebook is the **1024×1024** rung of a three-point input-resolution
study over an otherwise identical experiment:

| rung | input | scale applied to the stored 1024 px frame |
|---|---|---|
| `11_ssd-mn2-fpnlite_mc_phenobench_320_finetune` | 320×320 | 0.31 |
| `11_ssd-mn2-fpnlite_mc_phenobench_512_finetune` | 512×512 | 0.50 |
| `11_ssd-mn2-fpnlite_mc_phenobench_1024_finetune` | 1024×1024 | 1.00 (native) |

The dataset bundle stores frames at their **native 1024 px** precisely so the
resizer can be moved without re-exporting data, so all three rungs consume the
same `train.record` and differ only in `fixed_shape_resizer`.

Held fixed across the ladder: the COCO checkpoint, the anchor prior in
normalised coordinates, aspect ratios, matcher thresholds, augmentation, and
NMS. Changed: `image_size`, and the batch size that memory forces at this
resolution together with the learning rate implied by it (linear scaling rule —
LR scaled by the same factor as the batch, so the expected per-step parameter
update is comparable across rungs).

**`fpn_anchor_scale` is multiplied by 3.2 — and that is what keeps the
anchor prior *unchanged*.** It reads as the opposite of the plain ladder's "do
not retune the anchors", but it is the same rule applied to the opposite
parametrisation. `multiscale_anchor_generator` sizes anchors in **pixels**:
pyramid level *l* gets a base anchor of `anchor_scale × 2^l`, and the pyramid
spans levels 3–7 whatever the input resolution is. Holding `anchor_scale` at
1.0 would therefore shrink every anchor relative to the frame by exactly the
factor the input grew — the largest base anchor covers 0.40 of the
frame at 320 but only 0.125 here, while crops reach ≈0.50 of the frame.
That rung would measure a broken anchor prior, not a resolution effect. Scaling
by 1024/320 = 3.2 reproduces the 320 rung's anchor set **exactly in
normalised coordinates**, so resolution stays the only variable.
`fpn_scales_per_octave` is dimensionless and does not move.

What resolution does change is the density of the anchor grid: the finest
pyramid level (P3, stride 8) goes from 40×40 at 320 to 128×128 here,
which is the mechanism by which higher resolution is expected to help small
instances. The pyramid keeps all five levels, so the anchor count grows with it:
section 10b reads the exact figure off the exported graph.

> Note: the TF2 zoo *does* publish an SSD-MobileNetV2-FPNLite 640×640
> checkpoint, and this rung deliberately does not use it. Every rung of the
> ladder fine-tunes from the same 320×320 COCO checkpoint, because swapping the
> initialisation on one rung would confound the resolution effect with a
> pretraining effect. The cost is the same one the plain ladder pays: the
> backbone's learned scale priors are matched to a lower-resolution input, which
> is itself part of what the ladder measures.

## Output contract (`/kaggle/working`)

On completion the working directory holds exactly:

```
/kaggle/working/
├── manifest.json
└── finetune/
    ├── checkpoint/          # model-only ckpt-0, reusable as a model_path
    ├── saved_model/         # fp32 SavedModel for inference
    ├── graphs/              # training / validation curves (PDF)
    ├── pipeline.config      # as-run TFOD pipeline
    ├── best_metric.json
    └── metrics_history.json
```

## Session 2

This rung does not converge in one hosted session, so it is split into a chain
of notebooks. This one **resumes session 1**, which it imports as a
notebook input (`ssd-mn2-fpnlite_mc_phenobench_1024_finetune_1`) — so that session must have
finished and published a version before this one is run.


Resuming restores both halves — the checkpoint gives weights, optimizer and
step, the state file gives the best metric, the plateau / early-stopping
counters and the curves — so the chain trains under the *same stopping rule* as
one uninterrupted run. That is what keeps this rung comparable with 320 and 512,
which is the only reason the ladder means anything.

Whichever session converges is the one the PTQ / QAT notebooks must import: point
their `FINETUNE_DIR` at that session's slug, not at session 1's.

The companion PTQ / QAT notebooks consume this `finetune/` directory — its
`checkpoint/` + `pipeline.config` is a drop-in `model_path` — and write their own
`ptq/` and `qatN/` stage directories beside it, extending the same
`manifest.json`.

## 1 · Environment

Target runtime (Kaggle, "Pin to original environment" enabled):

- Python 3.10 · TensorFlow 2.11 · CUDA-enabled GPU

> **Kaggle inputs:** attach the **`ssd-mobilenet-v2-fpnlite-320x320-coco17-tpu-8`** model and the `mc-phenobench` / PhenoBench raw datasets as notebook inputs before running.

In [ ]:
!python -V

In [ ]:
!pip install --no-cache-dir --no-deps \
  tf_slim \
  pycocotools \
  lvis \
  contextlib2 \
  gin-config \
  tf-models-official==2.13.2 \
  tensorflow-model-optimization==0.7.5 \
  git+https://github.com/frdiener/agri-vision-edge.git

In [ ]:
import json
import shutil
from pathlib import Path
from dataclasses import asdict
import matplotlib.pyplot as plt

# setup_tensorflow_models() must run before anything imports object_detection.
from agri_vision_edge.third_party import setup_tensorflow_models
setup_tensorflow_models()

from agri_vision_edge.experiment import (
    ExperimentManifest,
    capture_environment,
)
from agri_vision_edge.experiment import (
    AugmentationConfig,
    FineTuneConfig,
)

# The shared trainer: one config object drives finetune / QAT + export.
from agri_vision_edge.tfod_trainer import (
    FinetuneRunConfig,
    TrainingControlConfig,
    run_finetune,
    write_pipeline,
    export_run,
)

# Curves are read back from the trainer's metrics_history.json (no TensorBoard).
from agri_vision_edge.evaluation.curves import (
    load_history_scalars,
    available_tags,
    plot_metric_curves,
    plot_loss_curves,
    plot_learning_rate,
    plot_steps_per_second,
    plot_map_curves,
    plot_recall_curves,
)

## 2 · Output paths

`tfod_trainer` writes its internal working tree (a `finetune/` pipeline dir and a
`train/` checkpoint dir) into a scratch location; we keep that in `/tmp` so the
published `/kaggle/working` contains only the `manifest.json` and the assembled
`finetune/` artifact (the exported fp32 base model).

In [ ]:
WORK = Path("/kaggle/working")

FINETUNE_DIR = WORK / "finetune"
GRAPHS_PATH = FINETUNE_DIR / "graphs"
MANIFEST_PATH = WORK / "manifest.json"

# NOT scratch on this rung. The run does not converge inside one hosted session,
# so the run dir is published and is what the next session resumes from: it
# holds the rolling `train/last/` checkpoint (weights + optimizer + step) and
# `train/trainer_state.json` (best metric, plateau counters, curves). Section 10
# deletes it again if -- and only if -- the run finished, so a converged run
# still publishes the usual {manifest.json, finetune/}.
RUN_DIR = WORK / "run"

# Resumes session 1 of 4, imported as a notebook input: its
# `run/` holds the rolling `train/last/` checkpoint and the
# `train/trainer_state.json` written when it hit its wall-clock budget.
RESUME_FROM = Path(
    "/kaggle/input/notebooks/freimutdiener/ssd-mn2-fpnlite-mc-phenobench-1024-finetune-1/run"
)

GRAPHS_PATH.mkdir(parents=True, exist_ok=True)

if RESUME_FROM is None:
    print("Cold start: no RESUME_FROM set.")
else:
    assert RESUME_FROM.is_dir(), f"no such notebook input: {RESUME_FROM}"
    shutil.copytree(RESUME_FROM, RUN_DIR, dirs_exist_ok=True)

    # Fail here rather than silently restarting from the COCO checkpoint under a
    # "resumed" label: `restore_weights` falls back to a cold start when it
    # finds no checkpoint, which would look like a working run and quietly throw
    # away the previous session.
    assert (RUN_DIR / "train" / "last").is_dir(), (
        f"{RESUME_FROM} holds no train/last/ checkpoint, so there is nothing to "
        "resume from -- the previous session died before its first evaluation."
    )
    print(f"Resuming from {RESUME_FROM}")

## 3 · Experiment manifest

The `ExperimentManifest` records environment, dataset, checkpoint, per-stage
configuration and metrics. It is written to `/kaggle/working/manifest.json` and
later extended in place by the QAT notebooks.

In [ ]:
manifest = ExperimentManifest(
    name="ssd-mobilenet-v2-fpnlite_mc_phenobench_1024x1024",
    task="object_detection",
)

manifest.set_environment(
    platform="kaggle",
    **capture_environment(),
)

## 4 · Fine-tuning configuration

`FineTuneConfig` is the high-level, framework-agnostic description of the run;
`tfod_trainer` renders it into a concrete TFOD `pipeline.config`. The anchor
section below is specific to this detector's anchor generator.

In [ ]:
config = FineTuneConfig(
    #
    # Optimization
    #

    # Batch size is set by activation memory at 1024×1024; the learning
    # rate follows it under the linear scaling rule so that the expected
    # per-step update stays comparable with the 320 rung (batch 16, 0.002).
    # If the session OOMs, halve both together -- the rule is what keeps
    # the rung comparable, not the specific numbers.
    batch_size=4,
    learning_rate_base=0.0005,
    warmup_learning_rate=0.00015,

    # Step cap only -- lr_plateau_exhausted_patience ends the run. Scaled
    # inversely with batch size so the cap corresponds to the same epoch
    # budget as the 320 rung. At 1024×1024 the Kaggle 12 h limit is the
    # binding constraint in practice; check the curves before assuming
    # the run converged rather than timed out.
    num_steps=320_000,
    warmup_steps=1500,


    image_size=1024,

    #
    # Anchor tuning -- FPN multiscale anchor generator
    #
    # The FPNLite head uses a `multiscale_anchor_generator`, so the classic
    # `min_scale` / `max_scale` sweep does not apply. Anchors are sized from a
    # single base `anchor_scale` (reduced from the COCO default of 4.0) with
    # `scales_per_octave` steps per pyramid level. At the 320 rung's 1.0
    # -- 3.2 here, see the rescale note below -- the level-3 anchors
    # are small enough for the densely packed weeds while the upper pyramid
    # levels still cover crops that fill most of a tile -- on the tiled
    # crop+weed mix, ~97% of crops and ~96% of weeds match at IoU 0.4.

    # Multiplied by 1024/320 = 3.2 on purpose, and *not* to retune
    # anything: `multiscale_anchor_generator` sizes anchors in pixels
    # (anchor_scale * 2**level) over a level 3-7 pyramid whose extent does
    # not depend on the input size. Holding this at 1.0 would shrink
    # every anchor relative to the frame by exactly the factor the input
    # grew, so the rung would measure a broken anchor prior instead of a
    # resolution effect. At 3.2x the anchor set is identical to the
    # 320 rung's in normalised coordinates.
    fpn_anchor_scale=3.2,
    fpn_scales_per_octave=2,

    anchor_aspect_ratios=(
        0.33,
        0.5,
        1.0,
        2.0,
        3.0,
    ),

    #
    # Matcher thresholds -- relaxed from the COCO default of 0.5 to recover
    # small, densely packed instances.
    #

    matched_threshold=0.4,
    unmatched_threshold=0.3,

    #
    # Data augmentation -- tuned for top-down agricultural imagery.
    #

    augmentation=AugmentationConfig(
        # Crop
        random_crop=True,
        crop_min_object_covered=1.0,
        crop_min_area=0.6,
        crop_max_area=1.0,
        crop_overlap_thresh=0.3,

        # Geometric invariance (overhead views are rotation/flip symmetric)
        horizontal_flip=True,
        horizontal_flip_probability=0.5,
        vertical_flip=True,
        vertical_flip_probability=0.5,
        rotation90=True,
        rotation90_probability=0.5,

        # Scale / zoom invariance
        zoom_range=(0.8, 1.2),

        # Photometric augmentation
        brightness_max_delta=0.15,
        contrast_range=(0.8, 1.2),
        saturation_range=(0.8, 1.2),
        hue_max_delta=0.02,

        # Compression robustness (enable if deployment images are JPEG-encoded)
        # jpeg_quality_range=(50, 100),
    ),

    #
    # Non-maximum suppression
    #

    nms_score_threshold=0.05,
    nms_iou_threshold=0.5,
    max_detections_per_class=100,
    max_total_detections=100,
)

manifest.add_stage("finetune")

"defined"

## 5 · Dataset

A `tfod_trainer` *dataset bundle* is a directory holding `label_map.pbtxt`,
`train.record` and `val.record` — exactly the layout of the mounted
`mc-phenobench` dataset. The raw PhenoBench images are used only for
qualitative evaluation at the end.

In [ ]:
manifest.set_dataset(
    name="phenobench",
    train_split="train",
    validation_split="val",
    num_classes=2,
)

dataset_dir = Path("/kaggle/input/datasets/freimutdiener/mc-phenobench-no-partials")
dataset_raw_dir = Path("/kaggle/input/datasets/freimutdiener/phenobench-raw-dataset-v1-1-0/PhenoBench")

label_map_path = dataset_dir / "label_map.pbtxt"
test_imgs = list((dataset_raw_dir / "test" / "images").glob("*.png"))

print(f"{len(test_imgs)} test images loaded")

## 6 · Base model and run configuration

We fine-tune from the COCO-2017 **SSD MobileNet V2 FPNLite (320×320)** checkpoint in the TF
model-zoo layout (`pipeline.config` + `checkpoint/ckpt-0.*`) — the same
checkpoint as every rung of the ladder, restored into a 1024×1024 resizer.
Both the backbone and the FPN are fully convolutional, so the restore is
shape-compatible; only the pyramid's grids grow. The base model,
the dataset bundle and the `FineTuneConfig` above are combined into a single
`FinetuneRunConfig`; the trainer renders the as-run `pipeline.config` from them.

In [ ]:
model_name = "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8"

manifest.set_checkpoint(
    model_name=model_name,
    pretrained_dataset="coco17",
    source="tensorflow_model_zoo",
)

# Base model in the TF model-zoo layout (pipeline.config + checkpoint/ckpt-0.*).
MODEL_DIR = (Path("/kaggle/input/models/freimutdiener/"
                  "ssd-mobilenet-v2-fpnlite-320x320-coco17-tpu-8/tensorflow2/default/1")
             / model_name)

In [ ]:
run_config = FinetuneRunConfig(
    model_path=MODEL_DIR,
    dataset_bundle_path=dataset_dir,
    num_classes=2,
    output_dir=RUN_DIR,      # scratch; the published artifact is assembled into FINETUNE_DIR
    finetune=config,
    control=TrainingControlConfig(
        lr_plateau=True,
        lr_plateau_factor=0.5,
        lr_plateau_patience=15,
        lr_plateau_cooldown=5,
        lr_plateau_min_lr=1e-5,
        lr_plateau_restore_best=True,
        lr_plateau_exhausted_patience=2,
        # Stop gracefully, well inside the platform's 12 h session limit. A run
        # killed *by* that limit exports nothing -- the cells below never run,
        # and a batch "Save & Run All" fails the version and discards the output
        # with the checkpoints in it -- so this rung stops itself, publishes,
        # and is resumed in a following session. The budget covers training and
        # the evaluations interleaved with it, and is only checked at eval
        # boundaries, so this leaves room for one interval of overshoot plus
        # export, plotting and upload.
        max_runtime_hours=10.5,
        # ignore partials on eval to mirror PhenoBench grading scheme
        eval_ignore_partials = True,
    ),
)

manifest.update_stage("finetune", config=run_config.to_mapping())

# Render the as-run pipeline now so it can be inspected before training.
write_pipeline(run_config)

print("Pipeline:", run_config.pipeline_config_path)

## 7 · Training

`run_finetune` builds the detection model, restores the COCO checkpoint, and
runs the training loop with metric-based checkpointing and early stopping,
appending one record per logged step to `metrics_history.json` and saving the
best checkpoint (by validation mAP) under the scratch run directory.

In [ ]:
result = run_finetune(run_config)

In [ ]:
# Promote the metric files from the scratch run dir into the finetune artifact.
shutil.copy2(result.best_metric_path, FINETUNE_DIR / "best_metric.json")
shutil.copy2(result.history_path, FINETUNE_DIR / "metrics_history.json")

best = json.loads((FINETUNE_DIR / "best_metric.json").read_text())
print(
    f"Best {best['metric_name']}: {best['metric_value']:.5f} "
    f"at step {best['step']}"
)

manifest.update_stage("finetune", metrics={"best_metric": best})
manifest.update_stage(
    "finetune",
    artifacts={
        "best_metric_json": "finetune/best_metric.json",
        "metrics_history_json": "finetune/metrics_history.json",
    },
)

## 8 · Training and validation curves

`load_history_scalars` turns the flat `metrics_history.json` into a tidy
long-format frame, which the `plot_*` helpers render. Each figure is saved into
`finetune/graphs/` as a vector PDF suitable for inclusion in reports or a thesis.

In [ ]:
history_df = load_history_scalars(FINETUNE_DIR / "metrics_history.json")

print("Available metric tags:")
for tag in available_tags(history_df):
    print("-", tag)

In [ ]:
loss_fig, _ = plot_loss_curves(
    history_df, smoothing=0.6,
    save_path=GRAPHS_PATH / "training_loss_curves.pdf",
)
display(loss_fig)

In [ ]:
lr_fig, _ = plot_learning_rate(
    history_df, smoothing=0.6,
    save_path=GRAPHS_PATH / "learning_rate_schedule.pdf",
)
display(lr_fig)

In [ ]:
tput_fig, _ = plot_steps_per_second(
    history_df, smoothing=0.5,
    save_path=GRAPHS_PATH / "training_throughput.pdf",
)
display(tput_fig)

In [ ]:
map_fig, _ = plot_map_curves(
    history_df, smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_precision_curves.pdf",
)
display(map_fig)

In [ ]:
recall_fig, _ = plot_recall_curves(
    history_df, smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_recall_curves.pdf",
)
display(recall_fig)

In [ ]:
size_precision_fig, _ = plot_metric_curves(
    df=history_df,
    tags=[
        "DetectionBoxes_Precision/mAP (small)",
        "DetectionBoxes_Precision/mAP (medium)",
        "DetectionBoxes_Precision/mAP (large)",
    ],
    title="Validation Precision per Object Size",
    ylabel="mAP",
    smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_precision_size_curves.pdf",
)
display(size_precision_fig)

In [ ]:
size_recall_fig, _ = plot_metric_curves(
    df=history_df,
    tags=[
        "DetectionBoxes_Recall/AR@100 (small)",
        "DetectionBoxes_Recall/AR@100 (medium)",
        "DetectionBoxes_Recall/AR@100 (large)",
    ],
    title="Validation Recall per Object Size",
    ylabel="Average Recall",
    smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_recall_size_curves.pdf",
)
display(size_recall_fig)

## 9 · Export

`export_run` writes the best checkpoint to the standard TF model-zoo layout
directly into `finetune/` (`checkpoint/ckpt-0.*`, `pipeline.config`, `saved_model/`).
The exported `checkpoint/` + `pipeline.config` is itself a valid `model_path`,
which is exactly how the companion PTQ / QAT notebooks resume from this finetune.

In [ ]:
export_result = export_run(run_config, export_dir=FINETUNE_DIR)

print("finetune dir:", export_result.export_dir)
print("SavedModel:", export_result.saved_model_dir)
print("Checkpoint:", export_result.checkpoint)
print("Pipeline:  ", export_result.pipeline_config)

## 10 · Register artifacts and publish

In [ ]:
manifest.add_artifact("finetune/pipeline.config", artifact_type="pipeline_config", stage="finetune")
manifest.add_artifact("finetune/graphs", artifact_type="evaluation_plots", stage="finetune")
manifest.add_artifact("finetune/saved_model", artifact_type="tfod_saved_model", stage="finetune")
manifest.add_artifact("finetune/checkpoint", artifact_type="tfod_checkpoint", stage="finetune")

manifest.save(MANIFEST_PATH)

# A finished run publishes the usual {manifest.json, finetune/}. An unfinished
# one must also publish `run/`: that is the only thing a follow-up session can
# resume from, and its presence in the output is the signal that this rung is
# not converged yet.
if result.converged:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    print("Run finished; run dir dropped.")
else:
    print(
        "Run stopped on its wall-clock budget and is NOT converged.\n"
        f"Publishing {RUN_DIR.name}/ for the next session: attach this "
        "notebook's output and set RESUME_FROM in section 2."
    )

print("Published:")
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(WORK))

## 10b · Resolution cost

The point of this notebook is the trade-off, so it is measured rather than
asserted. Three quantities are reported:

- **anchor count** — read off the exported detector's `raw_detection_boxes`.
  SSD's feature maps scale with the input, so the anchor count grows roughly
  quadratically, and it drives both the box-decode and the NMS cost.
- **host GPU latency** — the exported fp32 SavedModel, timed over 20 calls.
  This is *not* the deployment number: it is a float32 graph on a datacentre
  GPU. It is included because it isolates the resolution scaling on identical
  code; the figures that enter the thesis come from `ave benchmark` on the
  i.MX targets, over the INT8 TFLite conversion of this artifact.
- **SavedModel size** — parameter count is resolution-independent for a fully
  convolutional detector, so this should stay essentially flat. If it does not,
  something other than the resizer changed.

Read this together with the per-size precision curves in section 8: those say
what the resolution buys, this says what it costs.


In [ ]:
import time

import numpy as np
import tensorflow as tf
from object_detection.utils import config_util

from agri_vision_edge.tfod.inference import load_saved_model

# Read the resolution back from the as-run pipeline rather than trusting the
# config object: this is what the exported graph will actually resize to.
pipeline = config_util.get_configs_from_pipeline_file(
    str(FINETUNE_DIR / "pipeline.config")
)
resizer = pipeline["model"].ssd.image_resizer.fixed_shape_resizer
resolution = resizer.height
assert resizer.height == resizer.width == config.image_size, (
    f"pipeline resizer {resizer.height}x{resizer.width} disagrees with "
    f"FineTuneConfig.image_size={config.image_size}"
)

detect_fn = load_saved_model(str(FINETUNE_DIR / "saved_model"))
dummy = tf.zeros([1, resolution, resolution, 3], dtype=tf.uint8)

for _ in range(3):          # warm-up: the first calls trace and allocate
    detections = detect_fn(dummy)

timings = []
for _ in range(20):
    start = time.perf_counter()
    detect_fn(dummy)
    timings.append(time.perf_counter() - start)
timings = np.asarray(timings) * 1e3

saved_model_bytes = sum(
    f.stat().st_size for f in (FINETUNE_DIR / "saved_model").rglob("*") if f.is_file()
)

cost = {
    "resolution": int(resolution),
    "pixels_vs_320": round((resolution / 320) ** 2, 2),
    "anchors": (
        int(detections["raw_detection_boxes"].shape[1])
        if "raw_detection_boxes" in detections
        else None
    ),
    "host_gpu_latency_ms_mean": round(float(timings.mean()), 2),
    "host_gpu_latency_ms_median": round(float(np.median(timings)), 2),
    "host_gpu_latency_ms_std": round(float(timings.std()), 2),
    "saved_model_mb": round(saved_model_bytes / 1e6, 1),
}

for key, value in cost.items():
    print(f"{key:28s} {value}")

# Recorded in the manifest so the three rungs of the ladder can be tabulated
# without re-running anything.
manifest.update_stage("finetune", metrics={"resolution_cost": cost})
manifest.save(MANIFEST_PATH)

## 11 · Qualitative evaluation

Predictions on unseen PhenoBench test images, using the exported fp32
SavedModel from `finetune/`.

In [ ]:
%matplotlib inline

from agri_vision_edge.tfod.inference import (
    load_saved_model,
    load_label_map,
    detect_image,
)

detect_fn = load_saved_model(str(FINETUNE_DIR / "saved_model"))
category_index = load_label_map(label_map_path)

for image_path in test_imgs[:10]:
    vis, _ = detect_image(
        detect_fn=detect_fn,
        image_path=image_path,
        category_index=category_index,
        image_size=config.image_size,
        score_threshold=0.5,
        max_boxes=60,
    )
    plt.figure(figsize=(16, 16))
    plt.imshow(vis)
    plt.axis("off")
    plt.show()
    plt.close()

## 12 · Discussion

This is the 1024×1024 rung of the resolution ladder. Read it against the
320 and the other rungs on three axes:

1. **Precision by object size** (section 8) — the `mAP (small)` curve is
   where added resolution should pay off, because a small instance gains
   linearly in pixels while the anchor grid it must be matched against
   gains quadratically in density.
2. **Cost** (section 10b) — anchor count and host latency, both expected to
   scale roughly with the pixel count (10.24× that of the
   320 rung).
3. **Deployment** — the fp32 host numbers do not decide anything on their
   own. The comparison that matters is INT8 TFLite on the i.MX targets, and
   higher resolution costs more there than on a GPU: NMS is host-side, and
   the anchor count drives it directly.

A larger input is only worth it if the small-object gain survives
quantisation and fits the latency budget; that is the question this ladder
exists to answer, and it cannot be settled from the fp32 curves alone.

Next steps:
- **Quantization-aware training:** the companion `qatN` notebooks resume from
  this `finetune/` export with a `qat_scheme` set.
- **INT8 TFLite deployment:** `notebooks/tflite_conversion.py` consumes the
  `finetune/`, `ptq/` or a `qatN/` export.